# AI Pricing Agent Tutorial

This notebook demonstrates how to build an intelligent AI agent that automatically collects pricing information from various shops using Groq for AI capabilities, browser automation for web scraping, and Mapbox API for location-based functionality.

## Overview

Our AI Pricing Agent provides:

✅ **Multi-Item Price Collection**: Accept and validate lists of items for batch processing  
✅ **AI-Powered Parsing**: Use Groq API to intelligently extract and normalize pricing data  
✅ **Location Intelligence**: Integrate Mapbox API for geographic location data  
✅ **Concurrent Processing**: Handle multiple shop sources simultaneously for efficiency  
✅ **Browser Automation**: Navigate and extract data from dynamic websites  
✅ **Robust Error Handling**: Implement retry logic with exponential backoff  
✅ **Structured Output**: Return comprehensive metadata with confidence scores  

## Installation

First, install the required dependencies:

In [ ]:
!pip install -r requirements.txt

## Setup Environment Variables

Set up your API keys (you can also use a .env file):

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Or set them directly (uncomment and add your keys)
# os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'
# os.environ['MAPBOX_API_KEY'] = 'your_mapbox_api_key_here'  # Optional

# Verify keys are set
if os.getenv('GROQ_API_KEY'):
    print("✅ GROQ_API_KEY is set")
else:
    print("❌ GROQ_API_KEY is missing - please set it before continuing")

if os.getenv('MAPBOX_API_KEY'):
    print("✅ MAPBOX_API_KEY is set")
else:
    print("⚠️ MAPBOX_API_KEY is missing - location features will be disabled")

## Basic Usage

Let's start with a simple example of collecting prices for a few items:

In [ ]:
from pricing_agent import get_pricing_data
import json

# Define items to search for
items = ["wireless headphones", "gaming laptop", "smartphone"]

print(f"🔍 Searching for prices of: {', '.join(items)}")
print("This may take a moment as we search multiple shops...\n")

# Get pricing data
result = await get_pricing_data(items)

# Display summary
print(f"✅ Search completed!")
print(f"📊 Found {result['prices_found']} prices from {result['shop_sources_used']} shops")
print(f"⏱️ Execution time: {result['execution_time_seconds']} seconds\n")

# Display detailed results
print(json.dumps(result, indent=2))

## Advanced Usage: Custom Configuration

For more control, you can create a PricingAgent instance directly:

In [ ]:
from pricing_agent import PricingAgent

# Create agent instance
agent = PricingAgent()

# Validate items (this filters out invalid entries)
items = ["bluetooth speaker", "", "tablet", "   "]  # Note: empty strings will be filtered
validated_items = agent.validate_items(items)
print(f"Original items: {items}")
print(f"Validated items: {validated_items}")

# Collect pricing data
result = await agent.collect_pricing_data(validated_items)

# Show summary statistics
print(f"\n📈 Summary Statistics:")
print(f"   • Items requested: {result['items_requested']}")
print(f"   • Prices found: {result['prices_found']}")
print(f"   • Shop sources: {result['shop_sources_used']}")
print(f"   • Success rate: {(result['prices_found'] / (result['items_requested'] * result['shop_sources_used']) * 100):.1f}%")

## Understanding the Output

Let's examine the structure of the returned pricing data:

In [ ]:
# Get a sample result
sample_result = await get_pricing_data(["wireless mouse"])

print("📋 Response Structure:")
print(f"   • status: {sample_result['status']}")
print(f"   • timestamp: {sample_result['timestamp']}")
print(f"   • execution_time_seconds: {sample_result['execution_time_seconds']}")
print(f"   • items_requested: {sample_result['items_requested']}")
print(f"   • prices_found: {sample_result['prices_found']}")
print(f"   • shop_sources_used: {sample_result['shop_sources_used']}")

if sample_result['data']:
    sample_price = sample_result['data'][0]
    print("\n📦 Sample Price Data:")
    for key, value in sample_price.items():
        print(f"   • {key}: {value}")

## Error Handling Examples

The agent includes comprehensive error handling:

In [ ]:
# Test various error scenarios
test_cases = [
    ([], "Empty list"),
    ([""], "List with empty string"),
    ("not a list", "Wrong type"),
    (["   ", "", "  "], "All invalid items")
]

for test_input, description in test_cases:
    print(f"\nTesting: {description}")
    print(f"Input: {test_input}")
    try:
        result = await get_pricing_data(test_input)
        print(f"   ✅ Unexpected success: {result['items_requested']} items processed")
    except Exception as e:
        print(f"   ❌ Expected error: {e}")

## Testing Component Functionality

Let's test individual components of the agent:

In [ ]:
from pricing_agent import WebScraper, ShopSource, RetryHandler

# Test WebScraper
print("🕷️ Testing WebScraper...")
scraper = WebScraper()
await scraper.initialize()

shop = ShopSource("TestShop", "https://example.com", "/search?q={item}")
html_content = await scraper.scrape_shop(shop, "test item")
print(f"   Scraped content length: {len(html_content)} characters")

await scraper.cleanup()
print("   ✅ WebScraper test completed")

# Test RetryHandler
print("\n🔄 Testing RetryHandler...")
attempt_count = 0

async def failing_function():
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise Exception(f"Attempt {attempt_count} failed")
    return "Success!"

result = await RetryHandler.retry_with_backoff(failing_function, max_retries=5, base_delay=0.1)
print(f"   Result: {result} (after {attempt_count} attempts)")
print("   ✅ RetryHandler test completed")

## Architecture Overview

The AI Pricing Agent uses a modular architecture:

In [ ]:
from pricing_agent import ConfigManager, LocationService, GroqProcessor

print("🏗️ AI Pricing Agent Architecture:")
print("""
PricingAgent
├── ConfigManager      # API key validation and configuration
├── RetryHandler       # Exponential backoff retry logic
├── LocationService    # Mapbox API integration
├── GroqProcessor      # AI-powered price extraction
├── WebScraper        # Browser automation for web scraping
└── PriceData         # Structured data models
""")

# Show component status
print("📊 Component Status:")
try:
    config = ConfigManager()
    print("   ✅ ConfigManager: Initialized")
except Exception as e:
    print(f"   ❌ ConfigManager: {e}")

location_service = LocationService(os.getenv('MAPBOX_API_KEY'))
if location_service.api_key:
    print("   ✅ LocationService: Ready")
else:
    print("   ⚠️ LocationService: No API key (will use simulation mode)")

print("   ✅ WebScraper: Ready (simulation mode)")
print("   ✅ RetryHandler: Ready")
print("   ✅ Data Models: Ready")

## Performance Analysis

Let's analyze the performance characteristics of our agent:

In [ ]:
import time

# Test with different numbers of items
test_sizes = [1, 2, 3, 5]
performance_data = []

for size in test_sizes:
    items = [f"test_item_{i}" for i in range(size)]
    
    start_time = time.time()
    result = await get_pricing_data(items)
    end_time = time.time()
    
    performance_data.append({
        'items': size,
        'execution_time': result['execution_time_seconds'],
        'prices_found': result['prices_found'],
        'shops_per_item': result['shop_sources_used']
    })

print("📈 Performance Analysis:")
print("Items | Time(s) | Prices | Shops/Item")
print("-" * 40)
for data in performance_data:
    print(f"{data['items']:5} | {data['execution_time']:7.2f} | {data['prices_found']:6} | {data['shops_per_item']:10}")

# Calculate efficiency metrics
if len(performance_data) > 1:
    time_per_item = performance_data[-1]['execution_time'] / performance_data[-1]['items']
    print(f"\n⚡ Average time per item: {time_per_item:.2f} seconds")
    print(f"🎯 Concurrent processing: ✅ (multiple shops processed simultaneously)")

## Real-World Usage Scenarios

Here are some practical examples of how to use the pricing agent:

In [ ]:
# Scenario 1: Electronics Shopping
print("🛒 Scenario 1: Electronics Shopping")
electronics = ["4K monitor", "mechanical keyboard", "wireless mouse", "USB-C cable"]
electronics_result = await get_pricing_data(electronics)

print(f"Found {electronics_result['prices_found']} prices for {electronics_result['items_requested']} electronics items")

# Scenario 2: Home Office Setup
print("\n🏠 Scenario 2: Home Office Setup")
office_items = ["ergonomic chair", "standing desk", "desk lamp", "webcam"]
office_result = await get_pricing_data(office_items)

print(f"Found {office_result['prices_found']} prices for {office_result['items_requested']} office items")

# Scenario 3: Price Comparison for Specific Item
print("\n📱 Scenario 3: Price Comparison for iPhone")
iphone_result = await get_pricing_data(["iPhone 15"])

if iphone_result['data']:
    print("Price comparison across shops:")
    for price_data in iphone_result['data']:
        print(f"   {price_data['shop_name']}: ${price_data['price']} ({price_data['availability']})")
else:
    print("No pricing data found")

## Next Steps

This tutorial covered the basics of using the AI Pricing Agent. Here are some ways to extend it:

1. **Add More Shop Sources**: Extend the `shop_sources` list in the PricingAgent
2. **Custom Price Extraction**: Override the GroqProcessor for domain-specific parsing
3. **Caching**: Add caching to avoid repeated API calls
4. **Real Browser Automation**: Install browsers with `playwright install` for real web scraping
5. **Database Integration**: Store pricing data for historical analysis
6. **Price Alerts**: Set up notifications when prices drop below thresholds

## Resources

- [Groq API Documentation](https://console.groq.com/docs)
- [Mapbox API Documentation](https://docs.mapbox.com/)
- [Playwright Documentation](https://playwright.dev/python/)
- [Repository README](./README.md) for detailed documentation